# Module C: Customer Retention Intelligence & Survival Analysis
## RISK Enterprise Decision Intelligence Platform
**Role**: Customer Behavior Researcher & Machine Learning Engineer  
**Dataset**: Telco Customer Churn (`data/raw/telco_customer_churn.csv`)  
**Objective**: Empirical investigation of baseline retention, cross-sectional tenure cohorts, service interaction vulnerabilities, high-risk payment mechanisms, and supervised churn risk benchmarking.


### 1. Forensic Ingestion & Imputation
Loading raw customer records and applying domain-driven imputation (e.g. TotalCharges = $0.00 for tenure = 0 subscribers).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')

from src.loaders import load_telco_churn
from src.cleaning import clean_telco_churn
from src.metrics import calculate_retention_kpis, aggregate_churn_by_cohort
from src.visualization import set_reporting_style, PALETTE

set_reporting_style()

telco_raw = load_telco_churn('../data/raw/telco_customer_churn.csv')
df = clean_telco_churn(telco_raw)
print(f"Audited Subscriber Population: {len(df):,} customers across {df.shape[1]} features.")


### 2. Audited Retention & Exposure Scorecard
Disclosing the exact denominator (N = 7,043 eligible accounts) and calculating baseline churn and monthly revenue at risk.


In [ ]:
kpis = calculate_retention_kpis(df)
for k, v in kpis.items():
    if isinstance(v, float):
        print(f"{k:<30}: {v:,.2f}")
    else:
        print(f"{k:<30}: {v:,}")


### 3. Cross-Sectional Tenure Cohort Distribution
*Methodological Safeguard*: These cohorts represent cross-sectional survival slices across tenure bands, not longitudinal cohort tracking.


In [ ]:
cohort_summary = aggregate_churn_by_cohort(df, 'Tenure Cohort')
display(cohort_summary[['Tenure Cohort', 'Total_Customers', 'Churned_Customers', 'Churn_Rate_Pct', 'Retention_Rate_Pct']])


### 4. Contract Commitment & Payment Friction Analysis
Comparing Month-to-month contracts vs Long-term contracts and electronic check billing friction.


In [ ]:
contract_analysis = df.groupby('Contract', observed=False).agg(
    Subscribers=('customerID', 'count'),
    Churn_Rate_Pct=('Churn Binary', lambda x: x.mean() * 100),
    Avg_Monthly_Bill=('MonthlyCharges', 'mean')
).reset_index()
display(contract_analysis)

payment_analysis = df.groupby('PaymentMethod', observed=False).agg(
    Subscribers=('customerID', 'count'),
    Churn_Rate_Pct=('Churn Binary', lambda x: x.mean() * 100),
    Avg_Monthly_Bill=('MonthlyCharges', 'mean')
).reset_index()
display(payment_analysis.sort_values('Churn_Rate_Pct', ascending=False))


### 5. Service Combination Risk Matrix: Fiber Optic & Support Deficits
Evaluating the interaction between fiber optic broadband and technical support availability.


In [ ]:
fiber_subscribers = df[df['InternetService'] == 'Fiber optic']
tech_matrix = fiber_subscribers.groupby(['TechSupport', 'OnlineSecurity'], observed=False)['Churn Binary'].agg(
    Total='count',
    Churn_Rate_Pct=lambda x: x.mean() * 100
).reset_index()
display(tech_matrix)


### 6. Supervised Churn Risk Benchmarking (Leak-Free)
Training Logistic Regression and Random Forest models against a Dummy Majority baseline on a strict 80/20 stratified split.


In [ ]:
from src.retention_analysis import run_retention_analysis
results = run_retention_analysis()

print(f"Dummy Baseline Accuracy:     {results['dummy_baseline_accuracy']*100:.2f}%")
print(f"Logistic Regression ROC-AUC: {results['logistic_regression']['roc_auc']:.4f} (PR-AUC: {results['logistic_regression']['pr_auc']:.4f})")
print(f"Random Forest ROC-AUC:       {results['random_forest']['roc_auc']:.4f} (PR-AUC: {results['random_forest']['pr_auc']:.4f})")
